In [ ]:
import dgl
import torch
from dgl.data import CoraGraphDataset

print("Torch:", torch.__version__)
print("DGL:", dgl.__version__)
print("Environnement opérationnel !")

In [ ]:
# Chargement du dataset Cora
dataset = CoraGraphDataset()
graph = dataset[0]  # le graphe unique du dataset

# Affichage de base
print("Type de graphe :", type(graph))
print("Résumé du graphe :", graph)

## Structure du dataset Cora

Le dataset Cora est un graphe de citations scientifiques utilisé comme référence pour la classification de noeuds dans les graphes.

### 1. Représentation du graphe

- **noeuds** : chaque noeud représente un article scientifique.
- **Arêtes** : chaque arête représente une citation entre deux articles. Le graphe peut être considéré comme non orienté, car chaque citation est modélisée de façon symétrique.

### 2. Attributs disponibles dans `graph.ndata`

- `feat` : matrice de caractéristiques de taille `(N, 1433)` où `N` est le nombre de noeuds.  
  Chaque vecteur de taille 1433 est binaire (valeurs 0 ou 1) et encode la présence ou l’absence de mots-clés dans l’article correspondant.  
  Cette matrice est très creuse (sparse) : la grande majorité des composantes sont nulles.

- `label` : vecteur de labels entiers de taille `(N,)` indiquant la catégorie scientifique de chaque article.  
  Il existe 7 classes différentes (étiquetées de 0 à 6), correspondant à des domaines de recherche.

- `train_mask`, `val_mask`, `test_mask` : masques binaires (vecteurs de booléens de taille `(N,)`) indiquant l'appartenance de chaque noeud à l’un des sous-ensembles utilisés pour :
  - l'entraînement du modèle (`train_mask`)
  - la validation des performances pendant l'entraînement (`val_mask`)
  - le test final du modèle (`test_mask`)

### Exemple :

- `graph.ndata['feat'][i]` : vecteur de caractéristiques du noeud `i`.
- `graph.ndata['label'][i]` : classe associée au noeud `i`.
- `graph.ndata['train_mask'][i]` : vaut `True` si le noeud `i` est utilisé pour l'entraînement.

Ces informations permettent d'entraîner un modèle de classification de noeuds, en exploitant à la fois les caractéristiques individuelles des articles et la structure du graphe de citations.


In [ ]:
#Exploration du graphe Cora

# Taille du graphe
print("Nombre de noeuds :", graph.num_nodes())
print("Nombre d’arêtes :", graph.num_edges())

#Vérification de la symétrie des arêtes
src, dst = graph.edges()

#Attributs disponibles pour les noeuds
print("Clés dans graph.ndata :", graph.ndata.keys())

#Dimensions des features (vecteurs de caractéristiques)
features = graph.ndata['feat']
print("Taille des features :", features.shape)  # (nb de noeuds, taille du vecteur)

# Affichage des labels de quelques noeuds
labels = graph.ndata['label']
print("Label du nœud 0 :", labels[0].item())

### Questions de compréhension

1. On souhaite savoir si le graphe Cora est orienté ou non. En observant les arêtes, comment peut-on utilisez la fonction `has_edges_between(dst, src)` pour répondre à cette question.

2. Pourquoi peut-on traiter ce graphe comme un graphe non orienté, même si DGL le représente comme un graphe orienté en interne ? Quel est l'impact de cette décision sur le fonctionnement d’un GCN ?


In [ ]:
# Q1 : le graphe est non orienté si chaque arête (u -> v) a son arête retour (v -> u)
src, dst = graph.edges()
symetrique = graph.has_edges_between(dst, src).all().item()
print("Toutes les arêtes ont leur arête retour :", symetrique)

### Réponses

**1.** `graph.edges()` donne deux tenseurs `src` et `dst`. Si pour chaque arête `src[i] -> dst[i]` l'arête inverse `dst[i] -> src[i]` existe aussi, alors `graph.has_edges_between(dst, src)` renvoie `True` partout. Si c'est le cas (ce qu'on observe sur Cora), le graphe est en pratique non orienté : chaque citation est stockée dans les deux sens.

**2.** DGL stocke toujours des arêtes orientées, un graphe non orienté est donc juste un graphe où chaque arête est présente dans les deux sens. Pour la classification, une citation crée un lien thématique dans les deux sens : un article et celui qu'il cite parlent en général du même sujet, donc on veut que l'information circule dans les deux directions.

Impact sur le GCN : la couche agrège les messages des voisins entrants. Avec des arêtes symétriques, chaque nœud reçoit l'information des articles qu'il cite **et** de ceux qui le citent. La matrice d'adjacence $A$ est symétrique, et la normalisation $\hat D^{-1/2}\hat A \hat D^{-1/2}$ du GCN (qui suppose un graphe non orienté) a bien du sens. Avec un graphe orienté, certains nœuds recevraient très peu d'information (voire aucune s'ils n'ont pas d'arête entrante, ce que `GraphConv` refuse par défaut).

In [ ]:
# Définition d’un réseau GCN pour la classification de nœuds

import torch.nn as nn
import torch.nn.functional as F
from dgl.nn import GraphConv

class GCN(nn.Module):
    def __init__(self, in_feats, hidden_size, num_classes):
        super().__init__()
        self.conv1 = GraphConv(in_feats, hidden_size)
        self.conv2 = GraphConv(hidden_size, num_classes)

    def forward(self, g, x):
        x = self.conv1(g, x)
        x = F.relu(x)
        x = self.conv2(g, x)
        return x

# Initialisation du modèle
in_feats = graph.ndata['feat'].shape[1]
num_classes = len(torch.unique(graph.ndata['label']))
model = GCN(in_feats=in_feats, hidden_size=64, num_classes=num_classes)

print(model)


In [ ]:
# Entraînement du modèle GCN (avec GPU si disponible)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilisation du device : {device}")

# Transfert des données sur le device
graph = graph.to(device)
features = graph.ndata['feat'].to(device)
labels = graph.ndata['label'].to(device)
train_mask = graph.ndata['train_mask'].to(device)
val_mask = graph.ndata['val_mask'].to(device)

# Transfert du modèle
model = model.to(device)

# Optimiseur
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

train_losses = []
val_accuracies = []

# Boucle d'entraînement
for epoch in range(1, 1001):
    model.train()
    logits = model(graph, features)
    loss = F.cross_entropy(logits[train_mask], labels[train_mask])

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Évaluation sur validation
    model.eval()
    with torch.no_grad():
        val_logits = logits[val_mask]
        val_pred = val_logits.argmax(dim=1)
        val_acc = (val_pred == labels[val_mask]).float().mean()

    train_losses.append(loss.item())
    val_accuracies.append(val_acc.item())


    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f} | Validation accuracy: {val_acc.item():.4f}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))

# Courbe de loss en échelle log
plt.subplot(1, 2, 1)
plt.semilogy(train_losses, label="Train Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (log)")
plt.title("Courbe de loss (échelle log)")
plt.grid(True, which="both")
plt.legend()

# Courbe de validation accuracy (échelle normale)
plt.subplot(1, 2, 2)
plt.plot(val_accuracies, label="Validation Accuracy", color='orange')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Courbe de précision (validation)")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Évaluation finale du modèle sur le jeu de test (GPU-compatible)

test_mask = graph.ndata['test_mask'].to(device)

model.eval()
with torch.no_grad():
    logits = model(graph, features)  # logits est déjà sur le bon device
    test_logits = logits[test_mask]
    test_labels = labels[test_mask]  # labels est déjà sur le bon device
    test_pred = test_logits.argmax(dim=1)
    test_acc = (test_pred == test_labels).float().mean()

print(f"Test accuracy : {test_acc.item():.4f}")




## Matrice de confusion

La matrice de confusion permet d’observer en détail la qualité des prédictions du modèle.  
Chaque ligne correspond à une **classe réelle**, chaque colonne à une **classe prédite**.

Une diagonale dominante indique que les prédictions sont majoritairement correctes.  
Les confusions fréquentes entre certaines classes peuvent refléter leur proximité thématique.


In [ ]:
# Matrice de confusion sur le jeu de test

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

model.eval()
with torch.no_grad():
    logits = model(graph, features)
    test_mask = graph.ndata['test_mask']
    y_true = labels[test_mask].cpu().numpy()
    y_pred = logits[test_mask].argmax(dim=1).cpu().numpy()

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')


## Visualisation des embeddings (t-SNE)

Cette visualisation utilise la réduction de dimension t-SNE pour projeter les **représentations intermédiaires** des nœuds (après la première couche du GCN) dans un plan 2D.

Chaque point correspond à un nœud du graphe, coloré selon sa classe réelle.

Un bon regroupement des couleurs indique que le GCN a appris à séparer les différentes classes dans l’espace des représentations, ce qui facilite la classification finale.

In [ ]:
# Visualisation des embeddings appris par le GCN avec t-SNE

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Extraction des représentations (sortie de la 1re couche)
with torch.no_grad():
    x = F.relu(model.conv1(graph, features)).cpu().numpy()
    y = labels.cpu().numpy()

# Réduction de dimension avec t-SNE
tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=42)
x_2d = tsne.fit_transform(x)

# Affichage
plt.figure(figsize=(8, 6))
plt.scatter(x_2d[:, 0], x_2d[:, 1], c=y, cmap='tab10', s=10)
plt.title("Visualisation des représentations intermédiaires du GCN (t-SNE)")
plt.axis('off')
plt.show()



## Travail : Influence des hyperparamètres

Le modèle GCN défini dans ce TP comporte plusieurs **hyperparamètres** importants :

- `hidden_size` : dimension de la couche cachée (par défaut 16)
- `lr` : taux d'apprentissage de l'optimiseur (par défaut 0.005)
- `num_epochs` : nombre d'itérations d'entraînement (par défaut 1000)
- choix du nombre de **couches**, de la **fonction d'activation**, etc.

### Objectif

Modifiez un ou plusieurs de ces hyperparamètres pour observer leur effet sur :

- la **rapidité de convergence**
- la **précision sur le jeu de validation**
- le **score final sur le test**

Dans un premier temps, on vous propose de vous concentrer sur la taille cachée `hidden_size`.

### Travail demandé

1. Choisissez **un hyperparamètre à faire varier** (par exemple `hidden_size` ∈ {8, 32, 128, 512}).
2. Pour chaque valeur de cet hyperparamètre, réentraînez le modèle et :
   - stockez l’évolution de la **loss d’entraînement** à chaque epoch ;
   - stockez l’évolution de la **précision de validation** à chaque epoch.
3. Réalisez une **visualisation comparative** :
   - courbes d’**accuracy de validation** en fonction des epochs pour les différentes valeurs testées ;
   - éventuellement, courbes de **loss d’entraînement**.
4. À partir de ces courbes, choisissez une valeur “raisonnable” de l’hyperparamètre (par exemple une `hidden_size` qui offre un bon compromis entre performance et stabilité).
5. Avec cette configuration choisie :
   - évaluer **une seule fois** l’accuracy sur l’**ensemble de test** ;
   - comparer ce score de test à vos meilleurs scores de validation.

### Idée à discuter

- Pourquoi l’ensemble de **test** est-il gardé de côté pendant la phase d’exploration des hyperparamètres, et utilisé uniquement pour l’évaluation finale ?




### Réponse : influence de `hidden_size`

On écrit d'abord deux petites fonctions réutilisées dans tout le reste du TP :
- `evaluer` calcule l'accuracy sur un masque (en mode `eval`, sans gradient) ;
- `entrainer` fait la boucle d'apprentissage et garde la loss et l'accuracy de validation à chaque epoch.

Remarque : dans les boucles du TP, l'accuracy de validation est calculée avec les logits obtenus en mode `train`. Pour le GCN ce n'est pas grave (pas de dropout), mais pour le GAT (dropout) c'est plus juste de refaire une passe en mode `eval`, c'est ce que fait `evaluer`.

In [ ]:
def evaluer(model, g, mask):
    # accuracy du modèle sur les noeuds du masque
    model.eval()
    with torch.no_grad():
        pred = model(g, g.ndata['feat']).argmax(dim=1)
    return (pred[mask] == g.ndata['label'][mask]).float().mean().item()


def entrainer(model, g, epochs=200, lr=0.01, weight_decay=0.0):
    # boucle d'entraînement simple, renvoie les courbes de loss et d'accuracy de validation
    x = g.ndata['feat']
    y = g.ndata['label']
    train_mask = g.ndata['train_mask']
    val_mask = g.ndata['val_mask']
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    losses, val_accs = [], []
    for epoch in range(epochs):
        model.train()
        logits = model(g, x)
        loss = F.cross_entropy(logits[train_mask], y[train_mask])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        val_accs.append(evaluer(model, g, val_mask))
    return losses, val_accs

In [ ]:
# On fait varier hidden_size, le reste est fixé (lr = 0.005, 200 epochs)
tailles = [8, 32, 128, 512]
resultats = {}

for h in tailles:
    torch.manual_seed(0)
    model = GCN(in_feats, h, num_classes).to(device)
    losses, val_accs = entrainer(model, graph, epochs=200, lr=0.005)
    resultats[h] = (losses, val_accs)
    print(f"hidden_size = {h:3d} | meilleure val acc = {max(val_accs):.4f}")

In [ ]:
# Courbes comparatives
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
for h in tailles:
    plt.semilogy(resultats[h][0], label=f"hidden = {h}")
plt.xlabel("Epoch")
plt.ylabel("Train loss (log)")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
for h in tailles:
    plt.plot(resultats[h][1], label=f"hidden = {h}")
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# On choisit la taille qui a la meilleure accuracy de validation
best_h = max(tailles, key=lambda h: max(resultats[h][1]))
print("hidden_size retenu :", best_h)

# On réentraîne avec cette valeur et on évalue UNE seule fois sur le test
torch.manual_seed(0)
model = GCN(in_feats, best_h, num_classes).to(device)
losses, val_accs = entrainer(model, graph, epochs=200, lr=0.005)
print(f"Meilleure val acc : {max(val_accs):.4f}")
print(f"Test acc          : {evaluer(model, graph, graph.ndata['test_mask']):.4f}")

### Analyse

- **Convergence** : plus `hidden_size` est grand, plus la loss d'entraînement descend vite (le modèle a plus de paramètres). Avec `hidden = 8` la convergence est plus lente.
- **Validation** : l'accuracy de validation monte vite puis plafonne autour de 0.78–0.80 pour toutes les tailles raisonnables. Une très grande taille (512) n'apporte presque rien et sur-apprend plus vite (loss train quasi nulle avec seulement 140 nœuds d'entraînement).
- **Choix** : une taille intermédiaire (32 ou 64) est un bon compromis : même performance que 512, moins de paramètres, courbes plus stables.
- Le score de test est en général proche de la meilleure val acc, un peu en dessous, car la valeur a été choisie *sur* la validation (biais d'optimisme).

*(Les valeurs exactes sont à relire sur vos propres courbes après exécution.)*

**Pourquoi garder le test de côté ?** Si on choisissait les hyperparamètres en regardant le test, on adapterait le modèle au jeu de test et le score obtenu serait trop optimiste : il ne mesurerait plus la capacité à généraliser à des données jamais vues. La validation sert à faire les choix, le test sert seulement à donner une estimation honnête à la fin.

## De GCN à GraphSAGE

In [ ]:
from dgl.nn import SAGEConv

# Détection du device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)

# Définition du modèle GraphSAGE avec agrégation "mean"
class GraphSAGE(nn.Module):
    def __init__(self, in_feats, hidden_size, num_classes):
        super().__init__()
        self.conv1 = SAGEConv(in_feats, hidden_size, feat_drop=0, aggregator_type='mean')
        self.conv2 = SAGEConv(hidden_size, num_classes, feat_drop=0, aggregator_type='mean')

    def forward(self, g, x):
        x = self.conv1(g, x)
        x = F.relu(x)
        x = self.conv2(g, x)
        return x

# Transfert du graphe et des données sur le device
graph = graph.to(device)
features = graph.ndata['feat'].to(device)
labels = graph.ndata['label'].to(device)
train_mask = graph.ndata['train_mask'].to(device)
val_mask = graph.ndata['val_mask'].to(device)
test_mask = graph.ndata['test_mask'].to(device)

# Instanciation du modèle sur le bon device
model = GraphSAGE(in_feats=features.shape[1], hidden_size=32, num_classes=num_classes).to(device)

# Optimiseur
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

train_losses = []
val_accuracies = []


# Entraînement
for epoch in range(1, 101):
    model.train()
    logits = model(graph, features)
    loss = F.cross_entropy(logits[train_mask], labels[train_mask])

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        val_logits = logits[val_mask]
        val_pred = val_logits.argmax(dim=1)
        val_acc = (val_pred == labels[val_mask]).float().mean()

    train_losses.append(loss.item())
    val_accuracies.append(val_acc.item())


    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f} | Validation accuracy: {val_acc.item():.4f}")

# Évaluation finale
model.eval()
with torch.no_grad():
    logits = model(graph, features)
    test_logits = logits[test_mask]
    test_labels = labels[test_mask]
    test_pred = test_logits.argmax(dim=1)
    test_acc = (test_pred == test_labels).float().mean()

print(f"Test accuracy : {test_acc.item():.4f}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))

# Courbe de loss en échelle log
plt.subplot(1, 2, 1)
plt.semilogy(train_losses, label="Train Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (log)")
plt.title("Courbe de loss (échelle log)")
plt.grid(True, which="both")
plt.legend()

# Courbe de validation accuracy (échelle normale)
plt.subplot(1, 2, 2)
plt.plot(val_accuracies, label="Validation Accuracy", color='orange')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Courbe de précision (validation)")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

## Travail : Influence du type d’agrégateur dans GraphSAGE

On fixe désormais les hyperparamètres suivants pour tous les modèles GraphSAGE :

- `hidden_size = 32`
- `lr = 0.01`
- `num_epochs = 50`

On ne change **que** le type d’agrégateur dans `SAGEConv` parmi :
`aggregator_type ∈ { "mean", "gcn", "pool", "lstm" }`.

### 1. Mise en place des expériences

1. Entraînez un modèle GraphSAGE pour chacun des agrégateurs suivants :
   - `mean`
   - `gcn`
   - `pool`
   - `lstm`

   en gardant :
   - `hidden_size = 32`
   - `lr = 0.01`
   - `num_epochs = 50`.

2. Pour chaque agrégateur, enregistrez :
   - l’évolution de la **loss d’entraînement**,
   - l’évolution de l’**accuracy de validation**,
   - l’**accuracy finale sur le test** (évaluée une seule fois à la fin).

### 2. Analyse de la capacité de généralisation

3. Comparez les agrégateurs en termes de :
   - meilleure **accuracy de validation** atteinte,
   - **stabilité** des courbes,
   - **accuracy sur le test**.

   Lequel semble le mieux **généraliser** sur Cora avec ces hyperparamètres ?  
   Justifiez votre réponse à partir des scores de validation et de test.

### 3. Cas particulier de l’agrégateur LSTM

4. Comparez brièvement le comportement de `lstm` aux agrégateurs `mean` et `gcn` :
   - que remarquez-vous sur la **loss d’entraînement** et l’**accuracy de validation** ?
   - comment se situe son **score de test** ?

5. En une ou deux phrases, proposez des pistes pour améliorer la généralisation de `lstm`
   (par exemple en ajustant la taille cachée, le taux d’apprentissage ou en ajoutant
   de la régularisation comme du `dropout` ou du `weight_decay`).



### Réponse : comparaison des agrégateurs

In [ ]:
class GraphSAGEAgg(nn.Module):
    # même GraphSAGE que plus haut, mais on peut choisir l'agrégateur
    def __init__(self, in_feats, hidden_size, num_classes, agg):
        super().__init__()
        self.conv1 = SAGEConv(in_feats, hidden_size, aggregator_type=agg)
        self.conv2 = SAGEConv(hidden_size, num_classes, aggregator_type=agg)

    def forward(self, g, x):
        x = F.relu(self.conv1(g, x))
        return self.conv2(g, x)


aggregateurs = ["mean", "gcn", "pool", "lstm"]
res_agg = {}

for agg in aggregateurs:
    torch.manual_seed(0)
    model = GraphSAGEAgg(in_feats, 32, num_classes, agg).to(device)
    losses, val_accs = entrainer(model, graph, epochs=50, lr=0.01)
    test_acc = evaluer(model, graph, graph.ndata['test_mask'])
    res_agg[agg] = (losses, val_accs, test_acc)
    print(f"{agg:5s} | meilleure val acc = {max(val_accs):.4f} | test acc = {test_acc:.4f}")

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
for agg in aggregateurs:
    plt.semilogy(res_agg[agg][0], label=agg)
plt.xlabel("Epoch")
plt.ylabel("Train loss (log)")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
for agg in aggregateurs:
    plt.plot(res_agg[agg][1], label=agg)
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Analyse

**Généralisation.** `mean` et `gcn` sont les agrégateurs qui généralisent le mieux sur Cora : courbes de validation stables et meilleurs scores de validation et de test. C'est logique : Cora est un graphe homophile (les voisins ont souvent la même classe), donc faire une simple moyenne des voisins est déjà un très bon résumé. `gcn` est très proche de `mean` (il inclut le nœud lui-même dans la moyenne, comme un GCN). `pool` (MLP + max) a plus de paramètres et est un peu moins stable.

**Cas du LSTM.** L'agrégateur `lstm` a beaucoup plus de paramètres. On observe que sa loss d'entraînement descend très vite (il apprend par cœur les 140 nœuds d'entraînement) alors que son accuracy de validation est plus basse et plus irrégulière : c'est du sur-apprentissage. Son score de test est en dessous de `mean`/`gcn`. En plus, un LSTM traite les voisins comme une *séquence* ordonnée alors que l'ensemble des voisins n'a pas d'ordre : DGL les mélange, ce qui rend le résultat moins stable. Il est aussi nettement plus lent.

**Pistes pour le LSTM.** Réduire la taille cachée, baisser le learning rate, et ajouter de la régularisation : `feat_drop` (dropout sur les features, par ex. 0.5) et un `weight_decay` (par ex. 5e-4) dans Adam. Un early stopping sur la validation aide aussi.

*(Les valeurs exactes sont à relire sur vos propres sorties après exécution.)*

## De GraphSAGE aux GAT

In [ ]:
from dgl.nn import GATConv

class GAT(nn.Module):
    def __init__(self, in_feats, num_hidden, num_classes, num_heads=8, dropout=0.2):
        super().__init__()

        # Couche 1 : multi-head attention
        self.gat1 = GATConv(
            in_feats=in_feats,
            out_feats=num_hidden,
            num_heads=num_heads,
            feat_drop=dropout,
            attn_drop=dropout,
            activation=F.elu
        )

        # Couche 2 : une seule tête d'attention
        self.gat2 = GATConv(
            in_feats=num_hidden * num_heads,
            out_feats=num_classes,
            num_heads=1,
            feat_drop=dropout,
            attn_drop=dropout,
            activation=None  # Pas de softmax ici
        )

    def forward(self, g, x):
        x = self.gat1(g, x)
        x = x.flatten(1)  # concatène les têtes
        x = self.gat2(g, x)
        return x.squeeze(1)  # supprime la tête unique


#Transfert des données sur le GPU (si disponible)
graph = graph.to(device)
features = graph.ndata['feat'].to(device)
labels = graph.ndata['label'].to(device)
train_mask = graph.ndata['train_mask'].to(device)
val_mask = graph.ndata['val_mask'].to(device)
test_mask = graph.ndata['test_mask'].to(device)

#Instanciation du modèle
model = GAT(in_feats=features.shape[1], num_hidden=8, num_classes=num_classes, num_heads=8).to(device)

#Optimiseur
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

train_losses = []
val_accuracies = []

for epoch in range(1, 301):
    model.train()
    logits = model(graph, features)
    loss = F.cross_entropy(logits[train_mask], labels[train_mask])

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = logits[val_mask]
        val_pred = val_logits.argmax(dim=1)
        val_acc = (val_pred == labels[val_mask]).float().mean()

    train_losses.append(loss.item())
    val_accuracies.append(val_acc.item())

    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f} | Validation accuracy: {val_acc.item():.4f}")

# Évaluation finale
model.eval()
with torch.no_grad():
    logits = model(graph, features)
    test_logits = logits[test_mask]
    test_labels = labels[test_mask]
    test_pred = test_logits.argmax(dim=1)
    test_acc = (test_pred == test_labels).float().mean()

print(f"Test accuracy : {test_acc.item():.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))

# Courbe de loss en échelle log
plt.subplot(1, 2, 1)
plt.semilogy(train_losses, label="Train Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (log)")
plt.title("Courbe de loss (échelle log)")
plt.grid(True, which="both")
plt.legend()

# Courbe de validation accuracy (échelle normale)
plt.subplot(1, 2, 2)
plt.plot(val_accuracies, label="Validation Accuracy", color='orange')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Courbe de précision (validation)")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()



### Questions sur le fonctionnement de GAT

1. **Objectif de l’attention**

   Quel est l’objectif du mécanisme d’attention dans un GAT ?  
   En quoi cette approche diffère-t-elle de l’agrégation de voisinage utilisée dans GCN ou GraphSAGE (où les voisins sont moyennés ou sommés avec des poids fixes) ?

2. **Multi-head attention**

   Pourquoi utilise-t-on plusieurs "têtes" (`num_heads`) dans la première couche GAT ?  
   Quel est l’effet attendu de la concaténation des sorties multi-têtes sur la représentation finale des nœuds ?

3. **Influence du nombre de têtes d’attention à capacité fixe**

On souhaite étudier l’effet du nombre de têtes d’attention `num_heads` sans changer
la capacité totale du modèle.

1. Modifiez le modèle GAT pour fixer une **dimension cachée totale** `total_hidden_dim`
   (par exemple `128`), et ajuster automatiquement la dimension **par tête** :
   $$
   \text{hidden\_per\_head} = \frac{\text{total\_hidden\_dim}}{\text{num\_heads}}
   $$
   de sorte que, après concaténation des têtes, on obtienne toujours une représentation
   de taille `total_hidden_dim`, quel que soit `num_heads`.

2. En utilisant ce modèle à capacité fixe, faites varier `num_heads` (par exemple
   `num_heads ∈ {1, 2, 4, 8, 16}`) en gardant tous les autres hyperparamètres identiques
   (`total_hidden_dim`, `lr`, `weight_decay`, `num_epochs`, etc.).

   Pour chaque valeur de `num_heads` :
   - enregistrez l’évolution de l’**accuracy de validation**,
   - mesurez l’**accuracy sur le test** (une seule fois à la fin).

3. Analysez les résultats obtenus :
   - comment évoluent les **meilleurs scores de validation** et de **test** lorsque `num_heads` augmente,
     à capacité totale fixe ?
   - observe-t-on un intérêt à utiliser plusieurs têtes plutôt qu’une seule,
     même quand la dimension totale du modèle reste la même ?
   - jusqu’à quel point l’augmentation du nombre de têtes semble bénéfique sur ce problème ?


### Réponses

**1. Objectif de l'attention.** Dans un GCN ou un GraphSAGE, chaque voisin a un poids fixé par la structure du graphe (par ex. $1/\sqrt{d_i d_j}$ pour le GCN, $1/d_i$ pour la moyenne). Dans un GAT, le poids $\alpha_{ij}$ de chaque voisin est **appris** : il est calculé à partir des features des deux nœuds puis normalisé par un softmax sur le voisinage. Le modèle peut ainsi donner plus d'importance aux voisins utiles et moins aux voisins « bruités » (par ex. un article cité qui n'est pas du même domaine).

**2. Multi-head.** Chaque tête apprend ses propres poids d'attention, donc peut regarder le voisinage d'une façon différente. Utiliser plusieurs têtes stabilise l'apprentissage (le résultat dépend moins d'une seule attention mal initialisée), un peu comme un ensemble de modèles. En concaténant les sorties des têtes, chaque nœud obtient une représentation plus riche qui combine plusieurs « points de vue » sur son voisinage. Pour la dernière couche on prend une seule tête (ou une moyenne) pour obtenir directement les scores des classes.

In [ ]:
class GATFixe(nn.Module):
    # GAT où la dimension totale après concaténation des têtes est fixée
    def __init__(self, in_feats, total_hidden_dim, num_classes, num_heads, dropout=0.6):
        super().__init__()
        hidden_per_head = total_hidden_dim // num_heads
        self.gat1 = GATConv(in_feats, hidden_per_head, num_heads,
                            feat_drop=dropout, attn_drop=dropout, activation=F.elu)
        self.gat2 = GATConv(hidden_per_head * num_heads, num_classes, 1,
                            feat_drop=dropout, attn_drop=dropout)

    def forward(self, g, x):
        x = self.gat1(g, x).flatten(1)   # (N, total_hidden_dim)
        return self.gat2(g, x).squeeze(1)


liste_tetes = [1, 2, 4, 8, 16]
res_tetes = {}

for nh in liste_tetes:
    torch.manual_seed(0)
    model = GATFixe(in_feats, 128, num_classes, nh).to(device)
    losses, val_accs = entrainer(model, graph, epochs=200, lr=0.005, weight_decay=5e-4)
    test_acc = evaluer(model, graph, graph.ndata['test_mask'])
    res_tetes[nh] = (losses, val_accs, test_acc)
    print(f"heads = {nh:2d} (dim/tête = {128 // nh:3d}) | meilleure val acc = {max(val_accs):.4f} | test acc = {test_acc:.4f}")

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
for nh in liste_tetes:
    plt.plot(res_tetes[nh][1], label=f"{nh} têtes")
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(liste_tetes, [max(res_tetes[nh][1]) for nh in liste_tetes], "o-", label="meilleure val")
plt.plot(liste_tetes, [res_tetes[nh][2] for nh in liste_tetes], "s-", label="test")
plt.xscale("log", base=2)
plt.xlabel("Nombre de têtes")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Analyse

- À capacité totale fixée (128), passer de 1 tête à quelques têtes (4–8) améliore en général un peu la validation et le test, et surtout rend les courbes plus stables : plusieurs petites attentions indépendantes valent mieux qu'une seule grosse.
- Il y a donc bien un intérêt au multi-head même sans augmenter la taille du modèle : ce n'est pas seulement « plus de paramètres », c'est une diversité dans la façon d'agréger les voisins.
- Au-delà (16 têtes → seulement 8 dimensions par tête), le gain disparait voire s'inverse : chaque tête a trop peu de dimensions pour représenter quelque chose d'utile. Sur Cora, autour de 4 à 8 têtes semble être le bon compromis.

*(À confirmer avec vos chiffres : sur un si petit jeu de validation (500 nœuds) les écarts de l'ordre de 1 % peuvent venir du hasard, on peut relancer avec plusieurs seeds pour vérifier.)*

### Propose ton meilleur modèle selon tes critères.

Critère choisi : la **meilleure accuracy de validation**, le test n'est regardé qu'une fois à la fin.

D'après les expériences précédentes, le GAT régularisé est le plus performant. On reprend la configuration de l'article original du GAT (8 têtes de 8, dropout 0.6, weight decay 5e-4) et on ajoute un **early stopping** : on garde les poids de l'epoch qui a la meilleure validation.

In [ ]:
import copy

torch.manual_seed(0)
model = GATFixe(in_feats, 64, num_classes, num_heads=8, dropout=0.6).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

x = graph.ndata['feat']
y = graph.ndata['label']
best_val, best_poids = 0, None

for epoch in range(300):
    model.train()
    loss = F.cross_entropy(model(graph, x)[train_mask], y[train_mask])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # on garde les poids si la validation s'améliore
    val_acc = evaluer(model, graph, val_mask)
    if val_acc > best_val:
        best_val = val_acc
        best_poids = copy.deepcopy(model.state_dict())

model.load_state_dict(best_poids)
print(f"Meilleure val acc : {best_val:.4f}")
print(f"Test acc          : {evaluer(model, graph, test_mask):.4f}")

## Partie bonus – Changer d’échelle : le dataset **PubMed**

Dans cette section, nous vous proposons de réutiliser tout ce que vous avez vu sur Cora (GCN, GraphSAGE, GAT, réglage des hyperparamètres…) sur un **autre graphe de citations** : le dataset **PubMed**.

### Description du jeu de données PubMed

Le jeu de données PubMed (version citation network) est, comme Cora, un **graphe de citations d’articles scientifiques**, mais de plus grande taille.

- **Nœuds** : articles scientifiques publiés dans la base *PubMed*.
- **Arêtes** : relations de **citation** entre articles (souvent symétrisées dans les implémentations).
- **Tâche** : classification de nœuds (prédire le **thème principal** de l’article).
- **Caractéristiques des nœuds** :
  - vecteurs de caractéristiques réelles (par exemple basés sur des pondérations TF-IDF de mots),
  - dimension des features typiquement **plus faible que Cora**, mais les valeurs sont **continues** (et non binaires).
- **Labels** :
  - chaque nœud est associé à une des **3 classes** (thématiques médicales différentes).

Par rapport à Cora :

- le graphe est **plus grand** (plus de nœuds, plus d’arêtes),
- les features sont **continues** et non binaires,
- le problème reste une **classification supervisée de nœuds** avec séparation train / validation / test.

L’objectif est de voir si vos modèles et vos intuitions se **généralisent** lorsque l’on change de jeu de données et d’échelle.

---

### Consignes

1. **Charger** le dataset PubMed et récupérer :
   - le graphe `pubmed_graph`,
   - les tenseurs `features`, `labels`,
   - les masques `train_mask`, `val_mask`, `test_mask`.

2. **Explorer** le graphe :
   - nombre de nœuds et d’arêtes,
   - dimension des features,
   - nombre de classes et distribution des labels,
   - vérifier si le graphe est symétrique ou non.

3. **Réutiliser** (si vous le souhaitez) :
   - votre meilleur modèle entraîné sur Cora (GCN, GraphSAGE, GAT ou autre),
   - ou proposer une nouvelle architecture adaptée à PubMed.

4. **Comparer** qualitativement :
   - le comportement de l’apprentissage (courbes de loss / accuracy),
   - la difficulté apparente du problème,
   - l’influence de la taille du graphe et des hyperparamètres.

> Le but est ici exploratoire : vous avez à disposition tous les outils vus sur Cora, à vous de choisir comment les adapter à PubMed.

---

### Code de départ : import et accès aux données

Vous pouvez utiliser le code suivant comme point de départ dans une cellule *code*.

```python
import torch
from dgl.data import PubmedGraphDataset

# 1) Chargement du dataset PubMed
pubmed_dataset = PubmedGraphDataset()
pubmed_graph = pubmed_dataset[0]

print(pubmed_dataset)
print(pubmed_graph)


## Bonus PubMed : réponse

In [ ]:
from dgl.data import PubmedGraphDataset

# 1) Chargement
pubmed_dataset = PubmedGraphDataset()
pubmed_graph = pubmed_dataset[0].to(device)

features = pubmed_graph.ndata['feat']
labels = pubmed_graph.ndata['label']
train_mask = pubmed_graph.ndata['train_mask']
val_mask = pubmed_graph.ndata['val_mask']
test_mask = pubmed_graph.ndata['test_mask']

# 2) Exploration
print("Nombre de noeuds :", pubmed_graph.num_nodes())
print("Nombre d'arêtes  :", pubmed_graph.num_edges())
print("Dimension des features :", features.shape[1])
print("Nombre de classes :", pubmed_dataset.num_classes)
print("Répartition des labels :", torch.bincount(labels).tolist())
print("Train / val / test :", train_mask.sum().item(), val_mask.sum().item(), test_mask.sum().item())

src, dst = pubmed_graph.edges()
print("Graphe symétrique :", pubmed_graph.has_edges_between(dst, src).all().item())

In [ ]:
# 3) On compare un GCN simple et le GAT retenu sur Cora
pm_in = features.shape[1]
pm_classes = pubmed_dataset.num_classes

torch.manual_seed(0)
gcn_pm = GCN(pm_in, 64, pm_classes).to(device)
loss_gcn, val_gcn = entrainer(gcn_pm, pubmed_graph, epochs=200, lr=0.01, weight_decay=5e-4)

torch.manual_seed(0)
gat_pm = GATFixe(pm_in, 64, pm_classes, num_heads=8, dropout=0.6).to(device)
loss_gat, val_gat = entrainer(gat_pm, pubmed_graph, epochs=200, lr=0.01, weight_decay=5e-4)

print(f"GCN | meilleure val = {max(val_gcn):.4f} | test = {evaluer(gcn_pm, pubmed_graph, test_mask):.4f}")
print(f"GAT | meilleure val = {max(val_gat):.4f} | test = {evaluer(gat_pm, pubmed_graph, test_mask):.4f}")

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.semilogy(loss_gcn, label="GCN")
plt.semilogy(loss_gat, label="GAT")
plt.xlabel("Epoch")
plt.ylabel("Train loss (log)")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(val_gcn, label="GCN")
plt.plot(val_gat, label="GAT")
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Comparaison avec Cora

- **Taille** : PubMed est environ 7 fois plus grand que Cora (≈ 19 700 nœuds, ≈ 88 000 arêtes) mais avec seulement 500 features (TF-IDF continues) et 3 classes. Il est aussi symétrique.
- **Difficulté** : il n'y a que 60 nœuds d'entraînement (20 par classe), donc très peu de labels par rapport à la taille du graphe. Mais avec 3 classes la tâche reste abordable : on obtient des scores du même ordre que sur Cora (≈ 0.77–0.80 en test).
- **Apprentissage** : avec si peu de nœuds d'entraînement, la loss descend très vite ; la régularisation (`weight_decay`, dropout) est importante pour ne pas sur-apprendre. Le GAT n'apporte pas forcément un gros gain par rapport au GCN ici : sur un graphe homophile, une agrégation simple marche déjà bien.
- **Taille du graphe** : l'entraînement full-batch tient toujours largement sur le GPU (4 Go suffisent). Pour des graphes beaucoup plus grands il faudrait passer à du mini-batch avec échantillonnage de voisins (l'idée de GraphSAGE).

*(Valeurs à relire après exécution.)*